In [1]:
#r "C:\Users\user\Desktop\Lukyanov\practice2026\task17\bin\Debug\net10.0\task17.dll"
#r "nuget:ScottPlot,5.0.54"

using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.IO;
using System.Linq;
using System.Threading;
using ScottPlot;
using task17;

public class RepeatCommand : ILongCommand
{
    private readonly ICommand _inner;
    private readonly int _maxCalls;
    private int _calls;

    public bool IsCompleted
    {
    get { return _calls >= _maxCalls; }
    }

    public RepeatCommand(ICommand inner, int maxCalls)
    {
        _inner = inner;
        _maxCalls = maxCalls;
    }

    public void Execute()
    {
        if (!IsCompleted)
        {
            _inner.Execute();
            _calls++;
        }
    }
}

public class HeavyCommand : ILongCommand
{
    private int _rem;
    public bool IsCompleted => _rem <= 0;
    public HeavyCommand(int n) => _rem = n;
    public void Execute()
    {
        if (!IsCompleted)
        {
            _rem--;
            double x = 0;
            for (int i = 0; i < 100000; i++)
                x += Math.Sin(i) * Math.Cos(i);
        }
    }
}


Console.WriteLine("1.TestCommand (5 команд × 3 вызова)\n");


var scheduler = new RoundRobinScheduler();
var serverThread = new ServerThread(scheduler);
serverThread.Start();

for (int i = 1; i <= 5; i++)
{
    scheduler.Add(new RepeatCommand(new TestCommand(i), 3));
}

Console.WriteLine("Выполнение команд:\n");

while (scheduler.HasCommand())
{
    Thread.Sleep(50);
}

serverThread.Add(new HardStopCommand(serverThread));
serverThread.Join();

Console.WriteLine("\nПоток остановлен.\n");

Console.WriteLine("2. Замеры производительности: Один поток vs Round Robin\n");

var counts = new int[] { 1, 2, 4, 8, 16 };
var execPerCmd = 10;
var singleTimes = new double[counts.Length];
var rrTimes = new double[counts.Length];

for (int i = 0; i < counts.Length; i++)
{
    int n = counts[i];

    double singleTotal = 0;
    for (int run = 0; run < 3; run++)
    {
        var sw = Stopwatch.StartNew();
        for (int j = 0; j < n; j++)
        {
            var cmd = new HeavyCommand(execPerCmd);
            while (!cmd.IsCompleted) cmd.Execute();
        }
        sw.Stop();
        singleTotal += sw.Elapsed.TotalMilliseconds;
    }
    singleTimes[i] = singleTotal / 3;

    double rrTotal = 0;
    for (int run = 0; run < 3; run++)
    {
        var commands = new HeavyCommand[n];
        for (int j = 0; j < n; j++)
            commands[j] = new HeavyCommand(execPerCmd);

        var s = new RoundRobinScheduler();
        var st = new ServerThread(s);
        var sw = Stopwatch.StartNew();

        st.Start();
        for (int j = 0; j < n; j++)
            st.Add(commands[j]);

        while (!commands.All(c => c.IsCompleted))
            Thread.Sleep(1);

        sw.Stop();
        st.Add(new HardStopCommand(st));
        st.Join();
        rrTotal += sw.Elapsed.TotalMilliseconds;
    }
    rrTimes[i] = rrTotal / 3;

    Console.WriteLine($"Команд: {n,2} | Один поток: {singleTimes[i],8:F1} мс | Round Robin: {rrTimes[i],8:F1} мс");
}


var plot = new Plot();
plot.Title("Время выполнения: Один поток vs Round Robin");
plot.XLabel("Количество команд");
plot.YLabel("Время (мс)");

var xs = counts.Select(x => (double)x).ToArray();
plot.Add.Scatter(xs, singleTimes).Label = "Один поток";
plot.Add.Scatter(xs, rrTimes).Label = "Round Robin";
plot.ShowLegend();

var fn = "plot_long_ops.png";
plot.SavePng(fn, 800, 600);
display(HTML($"<img src='{fn}?t={DateTime.Now.Ticks}' width='700'/>"));

var reportPath = "report_long_ops.txt";
var sb = new System.Text.StringBuilder();

sb.AppendLine($"Дата: {DateTime.Now}");
sb.AppendLine();
sb.AppendLine("1. Round Robin scheduler обеспечивает чередование длительных команд,");
sb.AppendLine("   не блокируя поток на длительное время.");
sb.AppendLine("2. Команды ILongCommand автоматически возвращаются в очередь,");
sb.AppendLine("   пока IsCompleted == false.");
sb.AppendLine("3. HardStop корректно завершает поток после выполнения всех команд.");
sb.AppendLine("4. Время выполнения Round Robin сопоставимо с одним потоком,");
sb.AppendLine("   но позволяет прерывать и чередовать операции.");
File.WriteAllText(reportPath, sb.ToString());
Console.WriteLine($"\nОтчёт сохранён: {reportPath}");

Installed Packages ScottPlot, 5.0.54

Loading extensions from `C:\Users\user\.nuget\packages\skiasharp\2.88.9\interactive-extensions\dotnet\SkiaSharp.DotNet.Interactive.dll`

1.TestCommand (5 команд × 3 вызова)

Выполнение команд:

Поток 1 вызов 1
Поток 2 вызов 1
Поток 3 вызов 1
Поток 4 вызов 1
Поток 5 вызов 1
Поток 1 вызов 2
Поток 2 вызов 2
Поток 3 вызов 2
Поток 4 вызов 2
Поток 5 вызов 2
Поток 1 вызов 3
Поток 2 вызов 3
Поток 3 вызов 3
Поток 4 вызов 3
Поток 5 вызов 3

Поток остановлен.

2. Замеры производительности: Один поток vs Round Robin

Команд:  1 | Один поток:     25.2 мс | Round Robin:     28.7 мс
Команд:  2 | Один поток:     47.7 мс | Round Robin:     50.2 мс
Команд:  4 | Один поток:     96.2 мс | Round Robin:    113.2 мс
Команд:  8 | Один поток:    201.5 мс | Round Robin:    198.5 мс
Команд: 16 | Один поток:    372.6 мс | Round Robin:    381.5 мс



Отчёт сохранён: report_long_ops.txt
